## Mongodb

In [1]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')
print(f"✅ Model loaded | Vector size: {embedder.get_sentence_embedding_dimension()}")

d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...
✅ Model loaded | Vector size: 1024


In [2]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()

MONGO_URI = os.environ.get("MONGO_URI")

client = MongoClient(MONGO_URI)

db = client["RAG"]

collection = db["tariff_petition"]

# collection.insert_many(docs)

In [3]:
doc = collection.find_one()

print(type(doc["embedding"]))
print(len(doc["embedding"]))

<class 'list'>
1024


## Retriever

In [4]:
query = "What employee expenses has JUSNL projected?"

query_embedding = embedder.encode(
    query,
    normalize_embeddings=True
).tolist()

In [5]:
pipeline = [
    {
        "$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 100,
            "limit": 5
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": 1,
            "metadata": 1,
            "score": {
                "$meta": "vectorSearchScore"
            }
        }
    }
]

results = list(
    collection.aggregate(pipeline)
)

In [6]:
# results

In [7]:
pipeline = [
    {
        "$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 100,
            "limit": 5
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": 1,
            "metadata": 1,
            "score": {
                "$meta": "vectorSearchScore"
            }
        }
    }
]

In [8]:
results = list(
    collection.aggregate(pipeline)
)

for r in results:

    print(
        r["metadata"]["section_heading"]
    )

    print(
        r["score"]
    )

    print("-"*100)

4.5. Operation and Maintenance Expenses
0.8691350221633911
----------------------------------------------------------------------------------------------------
5.5. Operation and Maintenance Expenses
0.8629602193832397
----------------------------------------------------------------------------------------------------
4.5. Operation and Maintenance Expenses
0.8595227003097534
----------------------------------------------------------------------------------------------------
5.5. Operation and Maintenance Expenses
0.8591684103012085
----------------------------------------------------------------------------------------------------
5.5. Operation and Maintenance Expenses
0.8529952168464661
----------------------------------------------------------------------------------------------------


In [9]:
queries = [
    "What is JUSNL?",
    "Why has JUSNL filed this petition?",
    "What employee expenses are projected?",
    "What depreciation expenses are proposed?",
    "What return on equity is claimed?"
]

for query in queries:

    print("\n")
    print("="*100)
    print("QUESTION:", query)

    query_embedding = embedder.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": 3
            }
        },
        {
            "$project": {
                "_id": 0,
                "metadata": 1,
                "score": {
                    "$meta":
                    "vectorSearchScore"
                }
            }
        }
    ]

    results = list(
        collection.aggregate(
            pipeline
        )
    )

    for r in results:

        print(
            r["metadata"]["section_heading"]
        )

        print(
            round(
                r["score"],
                4
            )
        )

        print("-"*50)



QUESTION: What is JUSNL?
1.2. Profile of JUSNL
0.8495
--------------------------------------------------
1.1. Background
0.8323
--------------------------------------------------
1.1. Background
0.8264
--------------------------------------------------


QUESTION: Why has JUSNL filed this petition?
1.4. Rationale for filing of Instant Petition
0.8377
--------------------------------------------------
1.5. Contents of the Petition
0.8326
--------------------------------------------------
2.1. Present Approach
0.831
--------------------------------------------------


QUESTION: What employee expenses are projected?
4.5. Operation and Maintenance Expenses
0.8736
--------------------------------------------------
5.5. Operation and Maintenance Expenses
0.8523
--------------------------------------------------
5.5. Operation and Maintenance Expenses
0.8468
--------------------------------------------------


QUESTION: What depreciation expenses are proposed?
3.6. Depreciation
0.8721
-----

## Build Context Builder

In [10]:
# def build_context(results):

#     context = []

#     for r in results:

#         section = (
#             r["metadata"]
#             ["section_heading"]
#         )

#         pages = (
#             f"{r['metadata']['page_start']}"
#             f"-{r['metadata']['page_end']}"
#         )

#         context.append(

#             f"""
# SECTION: {section}

# PAGES: {pages}

# TEXT:
# {r['text']}
# """
#         )

#     return "\n\n".join(
#         context
#     )

def build_context(results):

    contexts = []

    for r in results:

        contexts.append(
            f"""
SOURCE

Section:
{r['metadata']['section_heading']}

Pages:
{r['metadata']['page_start']}-
{r['metadata']['page_end']}

Text:
{r['text']}
"""
        )

    return "\n".join(contexts)


def retrieve(
        query,
        top_k=5
):

    query_embedding = embedder.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": top_k
            }
        },
        {
            "$project": {
                "_id": 0,
                "text": 1,
                "metadata": 1,
                "score": {
                    "$meta":
                    "vectorSearchScore"
                }
            }
        }
    ]

    return list(
        collection.aggregate(
            pipeline
        )
    )

In [11]:
query = (
    "Why has JUSNL filed "
    "this petition?"
)

results = retrieve(query)

context = build_context(
    results
)

print(context[:5000])


SOURCE

Section:
1.4. Rationale for filing of Instant Petition

Pages:
12-
12

Text:
[Document: JUSNL | Section: 1.4. Rationale for filing of Instant Petition | Paragraphs: 1.4.1, 1.4.2, 1.4.3]

1.4. Rationale for filing of Instant Petition
1.4.1. Section 62 of the Electricity Act, 2003 requires the Transmission Licensee to
furnish details as may be specified by the SERC for determination of tariff. In
addition, as per the regulations issued by the Hon’ble Commission, JSEB or its
unbundled companies are required to file petition for all reasonable expenses which
they believe they would incur over the next financial year and seek the approval of
the Hon’ble Commission for the same in advance. The filing is to be done based on
the projections of expected costs and revenue.
1.4.2. The current petition has been prepared in accordance with the provisions of the
following Acts/ Policies/ Regulations:
a. The Electricity Act, 2003;
b. The National Electricity Policy;
c. The National Tariff Po

## LLM Connector

In [12]:
query = "Why has JUSNL filed this petition?"

context = build_context(
    retrieve(query)
)

def prompt_template(query, context):
    prompt = f"""
    You are a power sector regulatory analyst.

    Use ONLY the provided context.

    If answer is not present, say:
    'Information not available in retrieved documents.'

    For every answer provide:

    1. Answer
    Sources:
    * Regulatory Basis
    * Source Section
    * Page Number

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

In [13]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096}, {'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'openai/gpt-oss-20b', 'object': 'model', 'created': 1754407957, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'qwen/qwen3-32b', 'object': 'model', 'created': 1748396646, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 40960}, {'id':

In [14]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

llm = ChatGroq(
    api_key     = os.getenv("GROQ_API_KEY"),
        model_name  = "meta-llama/llama-4-scout-17b-16e-instruct",
    temperature = 0,

)
prompt = prompt_template(
    query,
    context)
response = llm.invoke(prompt)

print(response.content)

1. JUSNL has filed this petition to furnish details as required by the SERC for determination of tariff, as per Section 62 of the Electricity Act, 2003.

Sources:
* Regulatory Basis: The Electricity Act, 2003
* Source Section: 1.4. Rationale for filing of Instant Petition
* Page Number: 12


In [15]:
def ask(query, context):

    results = retrieve(query)

    context = build_context(results)

    prompt = prompt_template(query, context)

    answer = llm.invoke(prompt)
    # print(answer.content)

    return answer.content


ask("What petitions has JUSNL filed for FY 2023-24, FY 2024-25 and FY 2025-26?",context)



'1. JUSNL has filed the following petitions:\n   - Provisional True-up petition for FY 2023-24\n   - APR (Annual Performance Review) for FY 2024-25\n   - ARR (Annual Revenue Requirement) for FY 2025-26\n   - Tariff Proposal for FY 2025-26\n\nSources:\n* Regulatory Basis: \n* Source Section: 2.1. Present Approach, 1.5. Contents of the Petition\n* Page Number: 13, 12-13'

## Metadata Filter

In [16]:
from pymongo import MongoClient
from pydantic import BaseModel
from typing import Optional
import json
import re


# ====================================================
# BUILD DYNAMIC METADATA CATALOG
# ====================================================

def build_dynamic_catalog(collection):

    fields = [
        "document_type",
        "discom",
        "section_num",
        "main_section",
        "filing_year",
        "commission",
        "section_heading",
    ]

    catalog = {}

    for field in fields:

        values = collection.distinct(
            f"metadata.{field}"
        )

        # only keep fields with >1 values
        if len(values) > 1:

            catalog[field] = values

    return catalog


# ====================================================
# PYDANTIC MODEL
# ====================================================

class QueryFilters(BaseModel):

    semantic_query: str

    document_type: Optional[str] = None

    discom: Optional[str] = None

    filing_year: Optional[str] = None

    commission: Optional[str] = None
    section_num: Optional[str] = None
    main_section: Optional[str] = None
    section_heading : Optional[str] = None


# ====================================================
# PROMPT
# ====================================================

def build_filter_prompt(
        user_query,
        metadata_catalog
):

    return f"""
You are an expert query understanding engine for a Power Sector Regulatory Assistant.

Available metadata values:

{json.dumps(metadata_catalog, indent=2)}

Your task:

1. Understand the user's intent.
2. Extract ONLY metadata explicitly mentioned or strongly implied.
3. Never guess metadata.
4. If metadata is not present, do not include it.
5. Rewrite the question into a concise semantic search query.
6. Return ONLY valid JSON.
7. Do not add explanations.
8. Do not wrap output in markdown.

Metadata meanings:

- cost_head:
  Financial category such as employee_expense, depreciation, roe, interest, arr, capex.

- document_type:
  petition, tariff_order, regulation, trueup_order etc.

- filing_year:
  MUST exactly match one of the values in the available metadata above.
  Common formats: "FY24", "FY25", "FY26" or "FY 2024-25" etc.
  Always pick from the catalog values, never invent a value.

- discom:
  Utility name such as JUSNL.

- commission:
  Regulatory commission such as JSERC.

- section_num:
  Section identifier such as 1.1, 2.1, 5.5.

- main_section:
  Top-level chapter such as
  "1. Introduction",
  "2. Regulatory Framework",
  "5. ARR and Tariff Proposal".


- section_heading:
  Complete section title such as:

  "1.4. Rationale for filing of Instant Petition"

  "5.5. Operation and Maintenance Expenses"

  "3.8. Return on Equity"

Examples:

Question:
What employee expenses has JUSNL projected?

Output:

Question:
What does section 2.1 say?

Output:
{{
  "semantic_query": "section 2.1",
  "section_num": "2.1"
}}

Question:
Summarize the Introduction section.

Output:
{{
  "semantic_query": "introduction section summary",
  "main_section": "1. Introduction"
}}

Question:
What regulations govern transmission tariff?

Output:
{{
  "semantic_query": "transmission tariff regulations",
  "document_type": "regulation"
}}

Question:
What is JUSNL?

Output:
{{
  "semantic_query": "JUSNL profile background"
}}

User Question:

Question:
Show me the rationale for filing of instant petition

Output:
{{
  "semantic_query":
      "rationale for filing instant petition",

  "section_heading":
      "1.4. Rationale for filing of Instant Petition"
}}

{user_query}
"""


# ====================================================
# JSON PARSER
# ====================================================

def parse_json_response(text):

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if not match:

        raise ValueError(
            f"No JSON found:\n{text}"
        )

    return json.loads(
        match.group()
    )


# ====================================================
# EXTRACT FILTERS
# ====================================================

def extract_filters(

        user_query,

        llm,

        metadata_catalog
):
    # print("\nMETADATA CATALOG:\n", metadata_catalog)

    prompt = build_filter_prompt(

        user_query,

        metadata_catalog
    )

    response = llm.invoke(
        prompt
    )

    # print(
    #     "\nLLM RAW RESPONSE:\n",
    #     response.content
    # )

    parsed_json = parse_json_response(
        response.content
    )

    parsed_filters = QueryFilters(
        **parsed_json
    )

    return parsed_filters


# ====================================================
# BUILD MONGO FILTER
# ====================================================

def build_mongo_filter(
        parsed_filters
):

    mongo_filter = {}

    # cost head


    # document type

    if parsed_filters.document_type:

        mongo_filter[
            "metadata.document_type"
        ] = (
            parsed_filters.document_type
        )

    # discom

    if parsed_filters.discom:

        mongo_filter[
            "metadata.discom"
        ] = (
            parsed_filters.discom
        )

    # filing year

    if parsed_filters.filing_year:

        mongo_filter[
            "metadata.filing_year"
        ] = (
            parsed_filters.filing_year
        )

    # commission

    if parsed_filters.commission:

        mongo_filter[
            "metadata.commission"
        ] = (
            parsed_filters.commission
        )
    if parsed_filters.section_num:

        mongo_filter[
            "metadata.section_num"
        ] = (
            parsed_filters.section_num
        )

    if parsed_filters.main_section:

        mongo_filter[
            "metadata.main_section"
        ] = (
            parsed_filters.main_section
        )

    if parsed_filters.section_heading:
        mongo_filter[
            "metadata.section_heading"
            ] = (
                parsed_filters.section_heading
            )



    return mongo_filter


# ====================================================
# COMPLETE FUNCTION
# ====================================================

def get_query_filters(

        user_query,

        llm,

        collection
):

    metadata_catalog = (
        build_dynamic_catalog(
            collection
        )
    )

    parsed_filters = (
        extract_filters(

            user_query,

            llm,

            metadata_catalog
        )
    )
    # print("\nPARSED FILTERS:\n", parsed_filters)

    mongo_filter = (
        build_mongo_filter(
            parsed_filters
        )
    )


    return mongo_filter


# ====================================================
# TEST
# ====================================================


In [17]:
mongo_filter=get_query_filters(

    "What employee expenses has JUSNL projected?",

    llm,

    collection
)
mongo_filter

{'metadata.discom': 'JUSNL'}

In [18]:
print(mongo_filter)

print(
    collection.distinct(
        "metadata.cost_head"
    )
)

print(
    collection.count_documents(
        mongo_filter
    )
)

{'metadata.discom': 'JUSNL'}
['other']
157


## Hybrid Search (BM25/text search for mongodb + vector Search ) ---> RRF(Rank Retrieval Fusion Method for combining both searching)

In [19]:

from collections import defaultdict


## RRF Hybrid Retrieval
def hybrid_retrieve(
    query,
    top_k=5,
    vec_candidates=100,
    text_limit=50
):

    # ----------------------------------------
    # METADATA FILTER
    # ----------------------------------------

    mongo_filter = get_query_filters(query, llm, collection)
    print("\nGenerated Mongo Filter:", mongo_filter)

    # ----------------------------------------
    # Generate Query Embedding
    # ----------------------------------------

    query_embedding = embedder.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # ----------------------------------------
    # VECTOR SEARCH
    # ----------------------------------------

    vector_pipeline_config = {
        "index": "vector_index",
        "path": "embedding",
        "queryVector": query_embedding,
        "numCandidates": vec_candidates,
        "limit": text_limit,
    }

    # Only add filter if non-empty (empty dict breaks $vectorSearch)
    if mongo_filter:
        vector_pipeline_config["filter"] = mongo_filter

    vector_pipeline = [
        {
            "$vectorSearch": vector_pipeline_config
        },
        {
            "$project": {
                "_id": 1,
                "text": 1,
                "metadata": 1,
                "embedding": 1,
                "vec_score": {
                    "$meta": "vectorSearchScore"
                }
            }
        }
    ]

    vector_results = list(
        collection.aggregate(vector_pipeline)
    )

    # ----------------------------------------
    # TEXT SEARCH
    # BUG FIX: merge mongo_filter at top level, not as a nested "filter" key
    # ----------------------------------------

    text_query = {"$text": {"$search": query}}
    if mongo_filter:
        text_query.update(mongo_filter)   # merge metadata filters at top level

    text_results = list(
        collection.find(
            text_query,
            {
                "text": 1,
                "metadata": 1,
                "embedding": 1,
                "text_score": {
                    "$meta": "textScore"
                }
            }
        )
        .sort(
            [
                (
                    "text_score",
                    {
                        "$meta": "textScore"
                    }
                )
            ]
        )
        .limit(text_limit)
    )

    # ----------------------------------------
    # RRF FUSION
    # ----------------------------------------

    rrf_scores = defaultdict(float)

    docs = {}

    k = 60

    # Vector Search Ranking

    for rank, doc in enumerate(
        vector_results,
        start=1
    ):

        doc_id = str(doc["_id"])

        rrf_scores[doc_id] += (
            1 / (k + rank)
        )

        docs[doc_id] = doc

    # Text Search Ranking

    for rank, doc in enumerate(
        text_results,
        start=1
    ):

        doc_id = str(doc["_id"])

        rrf_scores[doc_id] += (
            1 / (k + rank)
        )

        if doc_id not in docs:

            docs[doc_id] = doc

    # ----------------------------------------
    # FINAL SORTING
    # ----------------------------------------

    ranked_docs = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    final_results = []

    for doc_id, score in ranked_docs[:top_k]:

        doc = docs[doc_id]

        doc["rrf_score"] = round(
            score,
            6
        )

        final_results.append(doc)

    return final_results


In [20]:
for idx in collection.list_indexes():
    print(idx)

SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('_fts', 'text'), ('_ftsx', 1)])), ('name', 'text_text'), ('weights', SON([('text', 1)])), ('default_language', 'english'), ('language_override', 'language'), ('textIndexVersion', 3)])


In [21]:
# collection.create_index([("text", "text")], default_language="english")

In [22]:
print(collection.index_information())

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'text_text': {'v': 2, 'key': [('_fts', 'text'), ('_ftsx', 1)], 'weights': SON([('text', 1)]), 'default_language': 'english', 'language_override': 'language', 'textIndexVersion': 3}}


In [23]:
results = hybrid_retrieve(
    query="What employee expenses has JUSNL projected?",
    top_k=5
)
\
for i, doc in enumerate(results, 1):

    print("\n" + "="*100)

    print("Rank:", i)

    print("RRF Score:",
          doc["rrf_score"])

    print("Text:")
    print(doc["text"][:500])


Generated Mongo Filter: {'metadata.discom': 'JUSNL'}

Rank: 1
RRF Score: 0.032266
Text:
[Document: JUSNL | Section: 4.5. Operation and Maintenance Expenses | Paragraphs: 4.5.1, 4.5.2]

4.5. Operation and Maintenance Expenses
4.5.1. The O&M expenses of JUSNL for the FY 2024-25 have been projected
considering the historical expenses and the projections in terms of capitalization
etc. The O&M expenses of FY 2023-24 are being used as base figures, which
are escalated to arrive at the future projections for FY 2024-25.
4.5.2. Operation and Maintenance expenses comprise of the following 

Rank: 2
RRF Score: 0.032002
Text:
[Document: JUSNL | Section: 5.5. Operation and Maintenance Expenses | Paragraphs: 5.5.5]

Annual Performance Review exercise and true up the employee cost and A&G
expenses on account of this variation, for the Control Period;
Note 2: Any variation due to changes recommended by the Pay Commission or
wage revision agreement, etc., will be considered separately by the Commiss

In [24]:
rrf_results = hybrid_retrieve(
    query="employee cost FY 2027-28",
    top_k=5
)

context = build_context(rrf_results)
# print(context[:5000])

ask("What petitions has JUSNL filed for FY 2023-24, FY 2024-25 and FY 2025-26?",context)


Generated Mongo Filter: {'metadata.filing_year': 'FY 2027-28'}


'1. JUSNL has filed the following petitions:\n   - Provisional True-up petition for FY 2023-24\n   - APR (Annual Performance Review) for FY 2024-25\n   - ARR (Annual Revenue Requirement) for FY 2025-26\n   - Tariff Proposal for FY 2025-26\n\nSources:\n* Regulatory Basis: \n* Source Section: 2.1. Present Approach, Section 1.5. Contents of the Petition\n* Page Number: 13, 12-13'

### MMR for diversity or reducing Redudancy of docs 

In [25]:
import numpy as np
from sentence_transformers import util
def mmr(
    query,
    documents,
    lambda_param=0.7,
    top_k=5
):

    query_embedding = embedder.encode(
    query,
      normalize_embeddings=True).tolist()



    selected = []

    candidates = documents.copy()

    while len(selected) < top_k and candidates:

        mmr_scores = []

        for doc in candidates:

            relevance = util.cos_sim(
                query_embedding,
                doc["embedding"]
            ).item()

            diversity = 0

            if selected:

                diversity = max(
                    util.cos_sim(
                        doc["embedding"],
                        selected_doc["embedding"]
                    ).item()
                    for selected_doc in selected
                )

            score = (
                lambda_param * relevance
                -
                (1 - lambda_param) * diversity
            )

            mmr_scores.append(score)

        best_idx = np.argmax(mmr_scores)

        best_doc = candidates.pop(best_idx)

        best_doc["mmr_score"] = float(
            mmr_scores[best_idx]
        )

        selected.append(best_doc)

    return selected

In [26]:
final_docs = mmr(
    query,
    rrf_results,
    lambda_param=0.7,
    top_k=5
)

In [27]:
# final_docs

In [28]:
for i, doc in enumerate(final_docs, 1):

    print("="*100)

    print("Rank:", i)

    print("MMR Score:",
          round(doc["mmr_score"], 4))

    print(doc["text"][:300])

In [29]:

query = "What is JUSNL?"

candidates = hybrid_retrieve(query=query, top_k=50 )
final_docs = mmr(
    query,
    candidates,
    lambda_param=0.7,
    top_k=5
)

final_docs
context = build_context(final_docs)
# print(context[:5000])

for doc in final_docs:

    print(
        doc["metadata"].get("main_section"),
        doc.get("rrf_score"),
        doc.get("mmr_score")
    )


Generated Mongo Filter: {'metadata.section_heading': '1.2. Profile of JUSNL'}
1. Introduction 0.032266 0.48925627470016475
1. Introduction 0.032522 0.2332525730133056
1. Introduction 0.032002 0.16951870918273923
1. Introduction 0.015625 0.12493797838687892


## Two-Stage Retrieval Funnel (Cross-Encoder Reranking)

Based on the blog pattern:
- **Stage 1** — Hybrid RRF (already built): casts a wide net, returns top-50
- **Stage 2** — Cross-encoder reranking: re-scores top-50 → returns top-10 with full query-document attention
- **Stage 3** — LLM listwise reranking: sees all 10 at once → returns final top-5 ordered by relative relevance

In [30]:
# Install if not already present
# !pip install sentence-transformers --quiet

from sentence_transformers import CrossEncoder
import numpy as np

print("Loading cross-encoder reranker model...")
# Best balance of speed and quality per the blog
# Alternatives: ms-marco-MiniLM-L-12-v2 (more accurate), ms-marco-TinyBERT-L-2-v2 (faster)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',device='cpu')
print("Reranker loaded.")

Loading cross-encoder reranker model...
Reranker loaded.


In [31]:
def stage2_crossencoder_rerank(query, candidates, top_k=10):
    """
    Stage 2: Cross-encoder reranking.

    Takes the hybrid_retrieve() candidates and re-scores each
    (query, chunk) pair using full cross-attention — the model
    sees query and document TOGETHER, unlike bi-encoder which
    encodes them separately.

    Architecture:
        Input: [CLS] query tokens [SEP] document tokens [SEP]
        → Transformer (full self-attention across ALL tokens)
        → [CLS] → Linear Head → sigmoid → relevance score (0 to 1)

    Args:
        query:      the user's question
        candidates: list of docs from hybrid_retrieve()
        top_k:      how many to keep after reranking

    Returns:
        Re-ordered list of top_k docs with 'cross_encoder_score' added
    """
    if not candidates:
        return candidates

    # Build (query, chunk_text) pairs — cross-encoder needs both together
    # This is the key difference: bi-encoder never sees them together
    pairs = [(query, doc["text"]) for doc in candidates]

    # Full transformer forward pass for each pair
    # Scores are raw logits — higher = more relevant
    scores = reranker.predict(pairs)

    # Attach score to each doc for inspection/debugging
    for doc, score in zip(candidates, scores):
        doc["cross_encoder_score"] = round(float(score), 4)

    # Sort by cross-encoder score (descending)
    reranked = sorted(
        candidates,
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return reranked[:top_k]

In [32]:
def stage3_llm_listwise_rerank(query, candidates, top_k=5):
    """
    Stage 3: LLM listwise reranking.

    Unlike Stage 2 (pairwise scoring), the LLM sees ALL candidates
    in a single prompt and produces a GLOBAL ordering.
    This is the only stage that can reason about relative relevance:
    'Document A is better than B because...'

    Only runs on Stage 2's top-k results (e.g. 10 docs),
    so LLM context window stays manageable.

    Args:
        query:      the user's question
        candidates: list of docs from stage2_crossencoder_rerank()
        top_k:      how many to keep in final output

    Returns:
        Final reordered list of top_k docs
    """
    if not candidates:
        return candidates

    if len(candidates) <= top_k:
        return candidates

    # Build the numbered list of documents for the prompt
    docs_text = ""
    for i, doc in enumerate(candidates):
        section = doc.get("metadata", {}).get("section_heading", "Unknown Section")
        snippet = doc["text"][:300].replace("\n", " ")
        docs_text += f"[{i+1}] Section: {section}\nText: {snippet}...\n\n"

    prompt = f"""You are a retrieval expert. Given a user query and a list of document chunks,
rank them from MOST to LEAST relevant to answer the query.

USER QUERY: {query}

DOCUMENTS:
{docs_text}

Return ONLY a JSON object with a single key "ranking" containing an ordered list of
document numbers (1-based), most relevant first. No explanation, just JSON.

Example format: {{"ranking": [3, 1, 4, 2, 5]}}
JSON:"""

    try:
        from langchain_core.messages import HumanMessage
        response = llm.invoke([HumanMessage(content=prompt)])
        response_text = response.content.strip()

        # Parse JSON safely
        import json, re
        json_match = re.search(r'\{.*?\}', response_text, re.DOTALL)
        if json_match:
            ranking_data = json.loads(json_match.group())
            ranking = ranking_data.get("ranking", [])

            # Convert 1-based indices to reordered doc list
            reordered = []
            for idx in ranking:
                if 1 <= idx <= len(candidates):
                    doc = candidates[idx - 1]
                    doc["llm_rank"] = len(reordered) + 1
                    reordered.append(doc)

            # Add any docs the LLM missed (safety fallback)
            ranked_indices = {r - 1 for r in ranking if 1 <= r <= len(candidates)}
            for i, doc in enumerate(candidates):
                if i not in ranked_indices:
                    reordered.append(doc)

            return reordered[:top_k]

    except Exception as e:
        print(f"LLM reranking failed ({e}), falling back to cross-encoder order")

    return candidates[:top_k]

In [33]:
def retrieve_rerank_pipeline(
    query,
    stage1_k=50,    # Wide net: hybrid RRF candidates
    stage2_k=10,    # Cross-encoder narrows to top-10
    stage3_k=5,     # LLM listwise narrows to final top-5
    use_mmr=True,   # Apply MMR for diversity before reranking
    use_llm_stage=True  # Toggle Stage 3 on/off (costs LLM call)
):
    """
    Full 3-stage retrieval funnel from the blog:

        MongoDB (all chunks)
            ↓ Stage 1: hybrid_retrieve() — RRF (fast, broad)
            top-{stage1_k} candidates
            ↓ [optional MMR for diversity]
            ↓ Stage 2: cross-encoder — full attention reranking
            top-{stage2_k} precise results
            ↓ Stage 3: LLM listwise — global relative ordering
            top-{stage3_k} final context
    """
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    # ── Stage 1: Hybrid RRF retrieval ──────────────────────────
    # print(f"\nStage 1 — Hybrid RRF retrieval (top-{stage1_k})...")
    stage1_results = hybrid_retrieve(query, top_k=stage1_k)
    # print(f"  Retrieved {len(stage1_results)} candidates")

    # ── Optional: MMR for diversity ────────────────────────────
    if use_mmr and len(stage1_results) > stage2_k:
        # print(f"  Applying MMR for diversity...")
        stage1_results = mmr(
            query,
            stage1_results,
            lambda_param=0.7,
            top_k=min(stage1_k, len(stage1_results))
        )

    # ── Stage 2: Cross-encoder reranking ───────────────────────
    # print(f"\nStage 2 — Cross-encoder reranking ({len(stage1_results)} → top-{stage2_k})...")
    stage2_results = stage2_crossencoder_rerank(query, stage1_results, top_k=stage2_k)
    # print(f"  Top-{stage2_k} cross-encoder scores:")
    for i, doc in enumerate(stage2_results[:5], 1):
        section = doc.get("metadata", {}).get("section_heading", "?")[:50]
        # print(f"    {i}. [{doc['cross_encoder_score']:+.4f}] {section}")

    # ── Stage 3: LLM listwise reranking ────────────────────────
    if use_llm_stage:
        # print(f"\nStage 3 — LLM listwise reranking ({stage2_k} → top-{stage3_k})...")
        final_results = stage3_llm_listwise_rerank(query, stage2_results, top_k=stage3_k)
        # print(f"  Final top-{stage3_k} after LLM ordering:")
    else:
        final_results = stage2_results[:stage3_k]
        # print(f"\nStage 3 skipped — using cross-encoder top-{stage3_k}")

    for i, doc in enumerate(final_results, 1):
        section = doc.get("metadata", {}).get("section_heading", "?")[:55]
        ce_score = doc.get("cross_encoder_score", "?")
        # print(f"    {i}. [CE: {ce_score:+.4f}] {section}")

    return final_results

In [34]:
def ask(query, context):
    """
    Full RAG pipeline with 3-stage retrieval funnel.
    Stage 1: Hybrid RRF → Stage 2: Cross-encoder → Stage 3: LLM listwise
    """
    final_docs = retrieve_rerank_pipeline(query)
    context = build_context(final_docs)
    prompt = prompt_template(query, context)
    answer = llm.invoke(prompt)
    print("\n" + "="*60)
    print("ANSWER:")
    print("="*60)
    print(answer.content)
    return answer.content

In [35]:
# ── Compare old pipeline vs new 3-stage funnel ──────────────
test_query = "What employee expenses has JUSNL projected?"

print("OLD PIPELINE (RRF + MMR only):")
print("-" * 50)
old_candidates = hybrid_retrieve(test_query, top_k=50)
old_final = mmr(test_query, old_candidates, lambda_param=0.7, top_k=5)
for i, doc in enumerate(old_final, 1):
    section = doc.get("metadata", {}).get("section_heading", "?")[:60]
    print(f"  {i}. [RRF: {doc.get('rrf_score', 0):.4f}] {section}")

print()
print("NEW PIPELINE (RRF + MMR + Cross-encoder + LLM listwise):")
print("-" * 50)
new_final = retrieve_rerank_pipeline(
    test_query,
    stage1_k=50,
    stage2_k=10,
    stage3_k=5,
    use_llm_stage=True
)

OLD PIPELINE (RRF + MMR only):
--------------------------------------------------

Generated Mongo Filter: {'metadata.discom': 'JUSNL'}
  1. [RRF: 0.0257] 4.5. Operation and Maintenance Expenses
  2. [RRF: 0.0318] 5.5. Operation and Maintenance Expenses
  3. [RRF: 0.0305] 3.5. Operation and Maintenance Expenses
  4. [RRF: 0.0320] 5.5. Operation and Maintenance Expenses
  5. [RRF: 0.0141] 5.5. Operation and Maintenance Expenses

NEW PIPELINE (RRF + MMR + Cross-encoder + LLM listwise):
--------------------------------------------------

Query: What employee expenses has JUSNL projected?

Generated Mongo Filter: {'metadata.discom': 'JUSNL'}


In [36]:
def display_result(result):

    print("\n" + "="*100)

    print("📌 ANSWER\n")

    print(result)

    print("\n" + "="*100)

In [37]:
context = build_context(new_final)
# print(context[:5000])

result=ask("What petitions has JUSNL filed for FY 2023-24, FY 2024-25 and FY 2025-26?",context)

print(type(result))
display_result(result)



Query: What petitions has JUSNL filed for FY 2023-24, FY 2024-25 and FY 2025-26?


ValidationError: 1 validation error for QueryFilters
filing_year
  Input should be a valid string [type=string_type, input_value=['FY 2023-24', 'FY 2024-25', 'FY 2025-26'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
benchmark_queries = [
    "What is JUSNL?",
    "Why has JUSNL filed this petition?",
    # "What regulations has JUSNL relied upon?",
    # "Why is JUSNL filing provisional true-up for FY 2023-24?",
    # "What petitions has JUSNL filed?",
    # "What employee expenses has JUSNL projected?",
    # "What A&G expenses has JUSNL projected?",
    # "What depreciation expenses has JUSNL proposed?",
    # "What return on equity has JUSNL claimed?",
    # "What interest on loan has JUSNL claimed?",
    # "What capital expenditure schemes has JUSNL proposed?",
    # "What ARR has JUSNL projected for FY 2025-26?",
    # "What transmission projects are proposed?",
    # "What are the major expenditure heads?",
    # "What are the major revenue sources?",
    # "Compare approved and projected employee expenses.",
    # "Compare FY 2024-25 and FY 2025-26 ARR.",
    # "Compare approved and actual capex.",
    # "What terminal benefits have been proposed?",
    # "How has RoE been calculated?"
]

for query in benchmark_queries:

    print("\n" + "="*100)
    print("QUESTION:", query)

    final_docs=retrieve_rerank_pipeline(
    query,
    stage1_k=50,
    stage2_k=10,
    stage3_k=5,
    use_llm_stage=True
        )



    context = build_context(final_docs)
    # print(context[:5000])

    result=ask(query, context)


    display_result(result)
    print("\n" + "="*100)





QUESTION: What is JUSNL?

Query: What is JUSNL?

Generated Mongo Filter: {'metadata.section_heading': '1.2. Profile of JUSNL'}

Query: What is JUSNL?

Generated Mongo Filter: {'metadata.section_heading': '1.2. Profile of JUSNL'}

ANSWER:
1. JUSNL is engaged primarily in the business of transmission of electricity.
 
Sources:
* Regulatory Basis: The Jharkhand State Electricity Reforms Revised Transfer Scheme, 2015 and the Electricity Act, 2003
* Source Section: 1.2. Profile of JUSNL
* Page Number: 9

📌 ANSWER

1. JUSNL is engaged primarily in the business of transmission of electricity.
 
Sources:
* Regulatory Basis: The Jharkhand State Electricity Reforms Revised Transfer Scheme, 2015 and the Electricity Act, 2003
* Source Section: 1.2. Profile of JUSNL
* Page Number: 9



QUESTION: Why has JUSNL filed this petition?

Query: Why has JUSNL filed this petition?

Generated Mongo Filter: {'metadata.section_heading': '1.4. Rationale for filing of Instant Petition'}

Query: Why has JUSNL fi

1. What is JUSNL?

2. Why has JUSNL filed this petition?

3. What regulations has JUSNL relied upon while filing the petition?

4. Why is JUSNL filing a provisional true-up for FY 2023-24?

5. What petitions has JUSNL filed for FY 2023-24, FY 2024-25 and FY 2025-26?

6. What employee expenses has JUSNL projected?

7. What depreciation expenses has JUSNL proposed?

8. What return on equity has JUSNL claimed?

9. What capital expenditure schemes has JUSNL proposed?

10. What ARR has JUSNL projected for FY 2025-26?

## Query Expanison or Rewriting

## Query Rewriting:

## Technique 1 — HyDE (Hypothetical Document Embedding)

In [38]:
def hyde_rewrite(query, llm):
    """
    Step 1: Generate a hypothetical answer.
    We only use this for SEARCHING — not as the final answer.
    """
    prompt = f"""You are an expert in Indian power sector regulation.

Write a detailed, factual-sounding passage that would answer this question.
Use domain-specific vocabulary: ₹ Cr, FY, O&M, ARR, JSERC, petitioner etc.
Write 3-4 sentences. Even if unsure, write something plausible.
Do NOT say "I don't know."

Question: {query}
Answer:"""

    response = llm.invoke(prompt)
    return response.content.strip()


def hyde_retrieve(query, embedder, llm, collection, top_k=10):
    """
    Step 2: Embed the hypothetical answer, not the raw question.
    Everything else is the same as standard retrieval.
    """
    # Generate fake answer
    hypothetical_answer = hyde_rewrite(query, llm)
    print(f"Hypothetical: {hypothetical_answer[:150]}...")

    # KEY DIFFERENCE: embed the answer, not the question
    search_vector = embedder.encode(
        hypothetical_answer,        # ← answer text
        normalize_embeddings=True
    ).tolist()

    # Standard vector search from here
    results = list(collection.aggregate([
        {"$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": search_vector,   # ← answer embedding
            "numCandidates": 100,
            "limit": top_k,
        }},
        {"$project": {"text": 1, "metadata": 1, "vec_score": {"$meta": "vectorSearchScore"}}}
    ]))

    return results

query  = "What employee expenses has JUSNL projected?"
hyde_retrieve(query, embedder, llm,collection)

Hypothetical: In its latest annual revenue requirement (ARR) petition filed with the Jharkhand State Electricity Regulatory Commission (JSERC), JUSNL (Jharkhand Sta...


[{'_id': ObjectId('6a270f3513f34b0acd35ea89'),
  'text': "[Document: JUSNL | Section: 1.1. Background | Paragraphs: 1.1.7, 1.1.8, 1.1.9]\n\nbeen transferred to Jharkhand Urja Sancharan Nigam Ltd (JUSNL).\n1.1.7. Section 62 of the Electricity Act 2003 requires the STU to furnish details as may be\nspecified by the Appropriate Commission for determination of tariff. In addition, as\nper the MYT Regulations issued by the Hon'ble Commission, JUSNL is required to\nfile for all reasonable expenses it believes it would incur over the next control\nperiod and seek the approval of the Hon'ble Commission for the same. The filing is\nto be done based on the projections of the expected revenue and costs, which\nshould be arrived at by a reasonable methodology adopted by the petitioner.\n1.1.8. The MYT Regulations notified by the Hon’ble Commission also mandates the filing\nof ARR for the MYT Control Period.\n1.1.9. The Govt. of India notified the Electricity Act, 2003 on 10thJune 2003 repealing th

## Technique 2 — Multi-Query

In [39]:
def multi_query_rewrite(query, llm, num_variants=4):
    """
    Step 1: Generate N rephrasings of the same question.
    """
    prompt = f"""Generate {num_variants} different search queries for this question.
Use different vocabulary, synonyms, and perspectives.
Think about how the answer might be written in a regulatory document.

Original: {query}

Output one query per line, no numbering:"""

    response = llm.invoke(prompt)
    variants = [
        line.strip()
        for line in response.content.strip().split("\n")
        if line.strip()
    ][:num_variants]

    return [query] + variants   # always include original


def multi_query_retrieve(query, embedder, llm, collection, top_k=10):
    """
    Step 2: Retrieve for each variant, fuse with RRF.
    """
    from collections import defaultdict

    all_queries = multi_query_rewrite(query, llm)
    print(f"Queries: {all_queries}")

    rrf_scores = defaultdict(float)
    docs = {}
    k = 60

    for i, q in enumerate(all_queries):
        # Original query gets slightly higher weight
        weight = 1.5 if i == 0 else 1.0

        q_vector = embedder.encode(q, normalize_embeddings=True).tolist()

        sub_results = list(collection.aggregate([
            {"$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": q_vector,
                "numCandidates": 50,
                "limit": 20,
            }},
            {"$project": {"text": 1, "metadata": 1, "vec_score": {"$meta": "vectorSearchScore"}}}
        ]))

        # RRF fusion — a doc appearing in multiple sub-results scores higher
        for rank, doc in enumerate(sub_results, start=1):
            doc_id = str(doc["_id"])
            rrf_scores[doc_id] += weight / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = doc

    # Sort by fused RRF score
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [docs[doc_id] for doc_id, _ in ranked[:top_k]]

multi_query_retrieve(query , embedder, llm , collection)

Queries: ['What employee expenses has JUSNL projected?', 'What forecasted expenditures has JUSNL allocated for staff costs?', 'What personnel-related expenses are projected by JUSNL in their financial outlook?', 'What employee outlays has JUSNL estimated in their budget projections?', 'What staff-related disbursements are anticipated by JUSNL according to their financial planning documents?']


[{'_id': ObjectId('6a270f3513f34b0acd35eaf8'),
  'text': '[Document: JUSNL | Section: 5.5. Operation and Maintenance Expenses | Paragraphs: 5.5.5]\n\nAnnual Performance Review exercise and true up the employee cost and A&G\nexpenses on account of this variation, for the Control Period;\nNote 2: Any variation due to changes recommended by the Pay Commission or\nwage revision agreement, etc., will be considered separately by the Commission;\nNote 3: Terminal Liabilities will be approved as per actual submitted by the\nTransmission Licensee or be established through actuarial studies.”\n5.5.5. The Petitioner has projected the employee cost for the FY 2025-26 by escalating\nthe projected employee cost (excluding the terminal benefits) estimated for FY\n2024-25 by the inflation factor of 6.09%. The same has been approved by the\nHon’ble Commission in the MYT Order for the 3rd Control Period. Also, it is\nsubmitted that JUSNL proposes to induct 96 junior managers and 10 managers\nduring the 

## Technique 3 — Step-Back Prompting

In [40]:
def stepback_rewrite(query, llm):
    """
    Step 1: Generate a broader, more abstract version of the question.
    """
    prompt = f"""Given this specific question, generate a broader abstract question
that covers the general topic. The abstract question retrieves foundational
context that helps answer the specific one.

Example:
  Specific: "What did JUSNL project for employee expenses in FY27?"
  Abstract: "What are O&M expense components and regulatory norms for power utilities?"

Now do this for:
  Specific: {query}
  Abstract:"""

    response = llm.invoke(prompt)
    return response.content.strip()


def stepback_retrieve(query, embedder, llm, collection, top_k=10):
    """
    Step 2: Retrieve for both abstract + specific, merge with RRF.
    Specific query gets higher weight (1.5x) since it's more targeted.
    """
    from collections import defaultdict

    abstract_query = stepback_rewrite(query, llm)
    print(f"Abstract: {abstract_query}")

    rrf_scores = defaultdict(float)
    docs = {}
    k = 60

    # Weight: specific query matters more than abstract
    for weight, q in [(1.5, query), (1.0, abstract_query)]:
        q_vector = embedder.encode(q, normalize_embeddings=True).tolist()

        sub_results = list(collection.aggregate([
            {"$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": q_vector,
                "numCandidates": 50,
                "limit": 20,
            }},
            {"$project": {"text": 1, "metadata": 1, "vec_score": {"$meta": "vectorSearchScore"}}}
        ]))

        for rank, doc in enumerate(sub_results, start=1):
            doc_id = str(doc["_id"])
            rrf_scores[doc_id] += weight / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = doc

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [docs[doc_id] for doc_id, _ in ranked[:top_k]]

## all three plug into your pipeline

In [42]:
def ask_with_rewriting(query, technique="hyde"):
    # Stage 1: retrieve with chosen technique
    if technique == "hyde":
        candidates = hyde_retrieve(query, embedder, llm, collection, top_k=50)
    elif technique == "multi":
        candidates = multi_query_retrieve(query, embedder, llm, collection, top_k=50)
    elif technique == "stepback":
        candidates = stepback_retrieve(query, embedder, llm, collection, top_k=50)

    # Stage 2 & 3: reranking — unchanged
    stage2 = stage2_crossencoder_rerank(query, candidates, top_k=10)
    final  = stage3_llm_listwise_rerank(query, stage2, top_k=5)

    # Generate answer — unchanged
    context = build_context(final)
    answer  = llm.invoke(prompt_template(query, context))
    return answer.content

In [ ]:
benchmark_queries = [
    "What is JUSNL?",
    "Why has JUSNL filed this petition?",
    # "What regulations has JUSNL relied upon?",
    # "Why is JUSNL filing provisional true-up for FY 2023-24?",
    # "What petitions has JUSNL filed?",
    # "What employee expenses has JUSNL projected?",
    # "What A&G expenses has JUSNL projected?",
    # "What depreciation expenses has JUSNL proposed?",
    # "What return on equity has JUSNL claimed?",
    # "What interest on loan has JUSNL claimed?",
    # "What capital expenditure schemes has JUSNL proposed?",
    # "What ARR has JUSNL projected for FY 2025-26?",
    # "What transmission projects are proposed?",
    # "What are the major expenditure heads?",
    # "What are the major revenue sources?",
    # "Compare approved and projected employee expenses.",
    # "Compare FY 2024-25 and FY 2025-26 ARR.",
    # "Compare approved and actual capex.",
    # "What terminal benefits have been proposed?",
    # "How has RoE been calculated?"
]

In [43]:
ask_with_rewriting(query="Why has JUSNL filed this petition?")

Hypothetical: JUSNL (Jharkhand State Electricity Company Limited) has filed this petition before the Jharkhand State Electricity Regulatory Commission (JSERC) seeki...


'1. JUSNL has filed this petition to seek approval for:\n   - Provisional True-up for FY 2023-24\n   - Annual Performance Review (APR) for FY 2024-25\n   - Annual Revenue Requirement (ARR) for FY 2025-26\n   - Tariff Proposal for FY 2025-26\n\nSources:\n* Regulatory Basis: The Electricity Act, 2003; JSERC (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020\n* Source Section: 1.4. Rationale for filing of Instant Petition; 1.5. Contents of the Petition; 2.1. Present Approach\n* Page Number: 12-13'